# Step 7: best normalisation, regularisation (point 1), fusion with clinical data (point 2) (GPU, ~6-8 h)
Everything uses **our pipeline** (`src/vitonly`): our normalisation, tumour crop and backbone. Akula's cache v2
uses one fixed normalisation; we compare against it later.
- **A. Best normalisation:** the two strongest ViT-only settings so far, 3 seeds each, ensemble, paired DeLong.
  The "(step 5)" one is reused from the repo (val 0.574) with `REUSE_STEP5`. The best on val is carried
  forward (`best_setting.json`); that choice is made on val, so say so in the report.
- **B. Regularisation (point 1)** of the best setting: frozen ViT (linear probe), 2 trainable blocks,
  largest-tumour-area slices, stronger augmentation, dropout 0.5 + weight decay. Each run changes one thing
  vs `base`, and train AUC is reported to show the overfitting gap.
- **C. Fusion (point 2):** the best setting + clinical tokens (HR, HER2, race, menopause, age, tumour-box
  volume) through attention (the F2 design, `configs/vo_fusion.yaml`), 3 seeds. AUC overall, per cohort and
  **within subtypes**, paired DeLong vs B1 (clinical + size).

Kaggle setup: Accelerator **GPU T4 x2** (one GPU is used), Internet **ON**. **+ Add Input**:
- the caches: `mnj-cache-ispy1`, `mnj-cache-ispy2`, `mnj-cache-duke`. Our pipeline normalises the images
  itself (`data.norm`), so any attached cache works; `mnj-cache2-*` (raw) or `mnj-cache-v2` are used
  automatically if attached;
- the code: the `mnj-code-vaib` Dataset (upload the new zip as a **new version** first) or the `GITHUB_TOKEN` secret.

Use **Save Version -> Save & Run All (Commit)**: these runs take hours and continue with the browser closed.
Results go to the Output tab under `results/<PERSON>/<step>/`. Keep this notebook's output: step 8 reads it.

In [ ]:
# ==== PARAMETERS (edit these) ====
REPO = "AkulaReddy/MRI_chemo_response"
GIT_TAG = "vaib"           # branch (or tag) to clone
PERSON = ""               # optional: "A", "B" or "C" -> results/<PERSON>/ (empty -> results/vaib/)
STEP = "step7"
CACHE_VERSION = None              # our pipeline: no cache-version requirement
CANDIDATES = {                    # ViT-only settings (overrides of configs/vo_base.yaml)
    "dinov2_b + sub + tumour crop (probe #1)": ["model.backbone=dinov2_b", "optim.lr_backbone=1.0e-5",
                                               "data.norm=sub", "data.crop=tumor", "model.pool=gap"],
    "mae + kinetic + tumour crop (step 5)":    ["model.backbone=mae", "data.norm=kinetic", "data.crop=tumor",
                                               "model.pool=gap"],
}
CAND_SEEDS = [0, 1, 2]
REUSE_STEP5 = True                # the "(step 5)" candidate: use its 3 seeds from results/B instead of retraining
RUN_SWEEP = True
SWEEP = {                         # name: overrides added to the best candidate; "base" = the best candidate itself
    "base":        [],
    "frozen":      ["model.train_blocks=0", "optim.freeze_epochs=0"],
    "blocks_2":    ["model.train_blocks=2"],
    "area_slices": ["data.slice_mode=area"],
    "strong_aug":  ["aug.rotate_deg=30", "aug.intensity=0.2", "aug.scale=[0.8, 1.2]"],
    "dropout_0.5": ["model.dropout=0.5", "optim.weight_decay=0.1"],
}
SWEEP_SEEDS = [0]
RUN_FUSION = True
FUSION_SEEDS = [0, 1, 2]
FUSION_EXTRA = []                 # extra overrides for the fusion model
FUSION_USE_BEST_SWEEP = False     # True: also add the best sweep entry's overrides (chosen on val)
TIME_BUDGET_H = 11.0
DEBUG = False
PUSH_RESULTS = False
NOTEBOOK_NAME = "step7"

In [ ]:
# ==== SETUP (do not edit): get the repo code, install requirements ====
import base64, json, os, shutil, subprocess, sys
if PERSON not in ("A", "B", "C"):           # only names the results folder
    PERSON = "vaib"
    print('PERSON not set: results go to results/vaib/ (set PERSON = "A", "B" or "C" to change)')
_tok = None
try:
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception as e:
    print("no GITHUB_TOKEN secret attached to this notebook (fine if the mnj-code-vaib Dataset is attached):",
          type(e).__name__)

def git(*args, cwd="/tmp/repo"):
    """git with the token as a one-off header (never stored in .git/config or the output)."""
    if not _tok:
        raise RuntimeError("no GITHUB_TOKEN secret: git push/pull is not available")
    hdr = "Authorization: Basic " + base64.b64encode(f"x-access-token:{_tok}".encode()).decode()
    r = subprocess.run(["git", "-c", f"http.extraHeader={hdr}", *args], cwd=cwd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(_tok, "***")
    if r.returncode:
        raise RuntimeError(f"git {args[0]} failed:\n{out}")
    return out

shutil.rmtree("/tmp/repo", ignore_errors=True)
if _tok:                                    # 1) clone the branch from GitHub
    git("clone", "--quiet", "--depth", "1", "--branch", GIT_TAG, f"https://github.com/{REPO}.git", "/tmp/repo", cwd="/tmp")
    print("code: GitHub", REPO, GIT_TAG, git("log", "-1", "--format=%h %s"))
else:                                       # 2) or a code Dataset (zip of the repo) attached as input
    import glob, zipfile
    hits = [h for d in range(1, 8) for h in glob.glob("/kaggle/input/" + "*/" * d + "src/vitonly/__init__.py")]
    zips = [z for d in range(1, 6) for z in glob.glob("/kaggle/input/" + "*/" * d + "mnj-code-vaib*.zip")]
    if not hits and zips:                   # Kaggle kept the uploaded zip as a file: unpack it
        zipfile.ZipFile(zips[0]).extractall("/tmp/code_zip")
        hits = glob.glob("/tmp/code_zip/**/src/vitonly/__init__.py", recursive=True)
    if not hits:
        print("attached inputs:")
        for root, dirs, files in os.walk("/kaggle/input"):
            if root.count("/") <= 5:
                print("  ", root, dirs[:6], files[:4])
            else:
                dirs[:] = []
        raise SystemExit(
            "\nThe notebook can't find the project code. Do ONE of these, then Run All again:\n"
            "  (a) Add-ons -> Secrets -> add GITHUB_TOKEN (a GitHub token with access to the repo) and switch it ON\n"
            "      for this notebook, or\n"
            "  (b) + Add Input -> Datasets -> Your Datasets -> mnj-code-vaib (made by uploading mnj-code-vaib.zip).")
    code_root = os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))
    shutil.copytree(code_root, "/tmp/repo")
    print("code: attached Dataset", code_root)
os.chdir("/tmp/repo")
sys.path.insert(0, "/tmp/repo")         # so `from src...` works in this kernel
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], check=True)
import torch; print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
# ==== HELPERS (do not edit): training runs, prediction files, comparisons ====
import glob, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src import evaluation as ev, stats
from src.train import find_caches
from src.data.metadata import patient_table
COH = ["ispy1", "ispy2", "duke"]
OUT = f"/kaggle/working/results/{PERSON}/{STEP}"
os.makedirs(OUT, exist_ok=True)
T0 = time.time()
RUNS_JSON = {}
pd.set_option("display.width", 250, "display.max_columns", 40)

def budget_left():
    return TIME_BUDGET_H - (time.time() - T0) / 3600

def say(msg):
    """Timestamped progress line (elapsed and remaining budget), flushed so it shows in the Kaggle log at once."""
    el = (time.time() - T0) / 3600
    print(f"[{time.strftime('%H:%M:%S')} | {el:.2f} h elapsed, {budget_left():.2f} h left] {msg}", flush=True)

def run_cmd(cmd):
    """Run a training subprocess and stream its output line by line into this notebook (subprocess.run alone
    shows nothing until it ends, and the child's stdout is block-buffered without PYTHONUNBUFFERED)."""
    print("\n" + " ".join(cmd), flush=True)
    t = time.time()
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise subprocess.CalledProcessError(p.returncode, cmd)
    say(f"finished in {(time.time() - t) / 60:.1f} min")

def check_caches(version):
    caches = find_caches(["/kaggle/input", "/kaggle/working"])
    info = {c: json.load(open(f"{d}/cache_meta.json")).get("builder_version", 1) for c, d in caches.items()}
    print("caches:", {c: f"builder v{v}: {caches[c]}" for c, v in info.items()})
    if version is not None:
        bad = {c: v for c, v in info.items() if v != version}
        assert not bad and len(info) == 3, (f"need cache builder v{version} for all 3 cohorts (attach mnj-cache-v2); "
                                            f"found {info}")
    return caches

def _copy(d, label):
    os.makedirs(f"{OUT}/{label}", exist_ok=True)
    for f in ("val_preds.csv", "metrics.json", "log.csv", "config_resolved.yaml", "export_preds.csv",
              "train_preds.csv", "encoder.json"):
        if os.path.exists(f"{d}/{f}"):
            shutil.copy(f"{d}/{f}", f"{OUT}/{label}/{f}")

def train(config, seed, sets=(), label=None, cohorts=None):
    """One src.train run (B3 / F2 ...) -> (run_dir, metrics). Its small files are copied to OUT/<label>/."""
    cmd = [sys.executable, "-m", "src.train", "--config", config, "--seed", str(seed), "--out", "/kaggle/working/runs"]
    if CACHE_VERSION is not None: cmd += ["--cache-version", str(CACHE_VERSION)]
    if cohorts: cmd += ["--cohorts", *cohorts]
    if DEBUG: cmd.append("--debug")
    if sets: cmd += ["--set", *sets]
    run_cmd(cmd)
    d = max(glob.glob("/kaggle/working/runs/*/"), key=os.path.getmtime).rstrip("/")
    if label: _copy(d, label)
    return d, json.load(open(f"{d}/metrics.json"))

def finetune(config, seed, sets=(), label=None, cohorts=None):
    """One src.vitonly.finetune run (our pipeline: ViT-only vo_base / fusion vo_fusion) -> run_dir."""
    cmd = [sys.executable, "-m", "src.vitonly.finetune", "--config", config, "--seed", str(seed),
           "--cache-root", "/kaggle/input", "/kaggle/working", "--out", "/kaggle/working/runs"]
    if cohorts: cmd += ["--cohorts", *cohorts]
    if DEBUG: cmd.append("--debug")
    if sets: cmd += ["--set", *sets]
    run_cmd(cmd)
    d = max(glob.glob("/kaggle/working/runs/*/"), key=os.path.getmtime).rstrip("/")
    if label: _copy(d, label)
    return d

def note_runs(key, dirs):
    """label -> run folder names, in OUT/runs.json: step 8 uses it to find the checkpoints."""
    RUNS_JSON[key] = [os.path.basename(d) for d in dirs]
    json.dump(RUNS_JSON, open(f"{OUT}/runs.json", "w"), indent=1)

def preds_of(dirs):
    fr = [ev.load_preds(f"{d}/val_preds.csv") for d in dirs]
    return fr[0] if len(fr) == 1 else ev.ensemble(fr)

def save_preds(label, df):
    os.makedirs(f"{OUT}/{label}", exist_ok=True)
    df.to_csv(f"{OUT}/{label}/val_preds.csv", index=False)
    return df

def repo_preds(pattern):
    hits = sorted(glob.glob(f"results/*/{pattern}/val_preds.csv"))
    return preds_of([os.path.dirname(h) for h in hits]) if hits else None

def train_auc(d):
    """AUC on the training patients from export_preds.csv (B3 exports after training): the overfitting gap."""
    f = f"{d}/export_preds.csv"
    if not os.path.exists(f): return np.nan
    e = pd.read_csv(f); e = e[e.split == "train"]
    return stats.auc(e.label, e.prob)

def vo_train_auc(d):
    """Training-set AUC of a src.vitonly run (metrics.json): the overfitting gap."""
    return json.load(open(f"{d}/metrics.json"))["val"].get("train_auc", np.nan)

def compare(preds, ref):
    """One row per model on the patients all models share: AUC (DeLong CI), AUPRC, Brier, ECE, the 90%-spec
    operating point (threshold chosen on these same val patients: optimistic), and the paired DeLong test
    and paired bootstrap of the AUC difference vs `ref`."""
    preds = ev.align({k: v for k, v in preds.items() if v is not None})
    rows = []
    for k, d in preds.items():
        r = {"model": k, **ev.summary_row(d.label, d.prob)}
        if k != ref and ref in preds:
            y = preds[ref].label
            dl = ev.delong(y, preds[ref].prob, d.prob)
            pb = stats.paired_bootstrap_auc_diff(y, preds[ref].prob, d.prob)
            r.update(diff_vs_ref=dl["diff"], delong_ci_lo=dl["ci"][0], delong_ci_hi=dl["ci"][1], delong_p=dl["p"],
                     boot_p=pb["p"])
        rows.append(r)
    cols = ["model", "n", "pcr", "auc", "ci_lo", "ci_hi", "auprc", "brier", "ece", "sens@90spec", "ppv@90spec",
            "npv@90spec", "diff_vs_ref", "delong_ci_lo", "delong_ci_hi", "delong_p", "boot_p"]
    t = pd.DataFrame(rows)
    return t[[c for c in cols if c in t]]

def subtype_wide(preds, by="subtype"):
    """AUC (95% CI) per subtype (or cohort) for every model, all on the same patients; n / pCR+ per group."""
    preds = ev.align({k: v for k, v in preds.items() if v is not None})
    tabs = {k: ev.subgroup_table(d, by).set_index(by) for k, d in preds.items()}
    first = next(iter(tabs.values()))
    out = pd.DataFrame({"n": first.n.astype(int), "pCR+": first.pcr.astype(int)})
    for k, t in tabs.items():
        out[k] = [f"{a:.3f} ({lo:.2f}-{hi:.2f})" if a == a else "n/a (one class)"
                  for a, lo, hi in zip(t.auc, t.ci_lo, t.ci_hi)]
    return out

B1_VAL = repo_preds("b1_lr_s0_*")                       # clinical + tumour size, logistic regression (val 0.653)
B0_VAL = repo_preds("b0_lr_s0_*")                       # clinical only (val 0.639)
AKULA_B3_V2 = repo_preds("b3_vit_s0_1019-0302_*")       # Akula's B3 on cache v2 (val 0.559)
B3_V1 = repo_preds("b3_vit_s0_1009-1616_*")             # B3 on the old cache (val 0.530)
F2_V1 = repo_preds("f2_attn_s0_1009-1605_*")            # fusion on the old cache (val 0.690)
print("reference predictions from the repo:", {k: v is not None for k, v in
      dict(B1=B1_VAL, B0=B0_VAL, B3_v2_Akula=AKULA_B3_V2, B3_v1=B3_V1, F2_v1=F2_V1).items()})

In [ ]:
caches = check_caches(CACHE_VERSION)

## A. Which normalisation / backbone is best? (ViT-only, 3 seeds each)

In [ ]:
import re
slug = lambda s: re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")
cand_runs, cand_preds = {}, {}
for name, sets in CANDIDATES.items():
    if REUSE_STEP5 and "(step 5)" in name and repo_preds("vo_base_mae_kinetic_tumor_s*") is not None:
        cand_preds[name] = repo_preds("vo_base_mae_kinetic_tumor_s*")
        print(f"{name}: reused the 3 step-5 seeds from results/B"); continue
    dirs = []
    for s in CAND_SEEDS:
        if budget_left() < 1.0:
            print(f"{name} seed {s} skipped: time budget"); continue
        dirs.append(finetune("vo_base", s, sets, f"step7_cand_{slug(name)}_s{s}"))
    cand_runs[name], cand_preds[name] = dirs, preds_of(dirs)
first = next(iter(cand_preds))
tab = compare(cand_preds, first)
tab.insert(1, "train_auc", [np.nanmean([vo_train_auc(d) for d in cand_runs.get(k, [])]) if cand_runs.get(k) else np.nan
                            for k in tab.model])
tab.to_csv(f"{OUT}/step7_candidates.csv", index=False); display(tab.round(3))
BEST = tab.sort_values("auc", ascending=False).model.iloc[0]
BEST_SETS = CANDIDATES[BEST]
save_preds("step7_vitonly_best_ensemble", cand_preds[BEST])
note_runs("vitonly_best", cand_runs.get(BEST, []))
json.dump({"name": BEST, "overrides": BEST_SETS}, open(f"{OUT}/best_setting.json", "w"), indent=1)
print(f"BEST (on val): {BEST} -> {BEST_SETS}")

## B. Point 1: regularisation of the best setting

In [ ]:
sweep = {}
if RUN_SWEEP:
    for name, extra in SWEEP.items():
        for s in SWEEP_SEEDS:
            reuse = cand_runs.get(BEST, [])
            if name == "base" and s in CAND_SEEDS and CAND_SEEDS.index(s) < len(reuse):
                sweep.setdefault(name, []).append(reuse[CAND_SEEDS.index(s)]); continue   # same run as candidate A
            if budget_left() < 0.9:
                print(f"{name} seed {s} skipped: time budget"); continue
            sweep.setdefault(name, []).append(finetune("vo_base", s, BEST_SETS + extra, f"step7_sweep_{name}_s{s}"))
    note_runs("sweep", [d for ds in sweep.values() for d in ds])
    sp = {n: preds_of(ds) for n, ds in sweep.items()}
    tab = compare(sp, "base")
    tab.insert(1, "overrides", [" ".join(SWEEP[n]) or "(best candidate)" for n in tab.model])
    tab.insert(2, "train_auc", [np.nanmean([vo_train_auc(d) for d in sweep[n]]) for n in tab.model])
    tab.insert(3, "gap", tab.train_auc - tab.auc)
    tab.to_csv(f"{OUT}/step7_sweep_summary.csv", index=False); display(tab.round(3))
    best_sweep = tab.sort_values("auc", ascending=False).model.iloc[0]
    note_runs("sweep_best", sweep[best_sweep]); save_preds("step7_sweep_best", sp[best_sweep])
    if not RUNS_JSON.get("vitonly_best") and sweep.get("base"):
        note_runs("vitonly_best", sweep["base"])   # best setting was reused from step 5 (no checkpoint in git):
                                                   # the sweep's base run is that same setting, trained here
    print(f"best on val: {best_sweep} {SWEEP[best_sweep]}. Trust it only if delong_p < 0.05 vs base; "
          "confirm with SWEEP_SEEDS = [0, 1, 2]. gap = train AUC - val AUC (overfitting).")

## C. Point 2: fusion with clinical data (our pipeline), overall / per cohort / within subtypes

In [ ]:
if RUN_FUSION:
    fsets = BEST_SETS + list(FUSION_EXTRA)
    if FUSION_USE_BEST_SWEEP and sweep:
        fsets += [x for x in SWEEP[best_sweep] if x not in fsets]
    dirs = []
    for s in FUSION_SEEDS:
        if budget_left() < 0.9:
            print(f"fusion seed {s} skipped: time budget"); continue
        dirs.append(finetune("vo_fusion", s, fsets, f"step7_fusion_s{s}"))
    note_runs("fusion", dirs)
    fus = save_preds("step7_fusion_ensemble", preds_of(dirs))
    json.dump({"name": BEST, "overrides": BEST_SETS, "fusion_overrides": fsets}, open(f"{OUT}/best_setting.json", "w"), indent=1)
    models = {"B1 clinical + size (LR)": B1_VAL, "B0 clinical (LR)": B0_VAL, "F2 fusion, old pipeline (repo)": F2_V1,
              "B3 ViT, cache v2 (Akula)": AKULA_B3_V2, f"ViT-only ours: {BEST}": cand_preds[BEST],
              f"Fusion ours ({len(dirs)} seeds)": fus}
    tab = compare(models, "B1 clinical + size (LR)")
    tab.to_csv(f"{OUT}/step7_fusion_summary.csv", index=False); display(tab.round(3))
    print("\nAUC within subtypes (95% CI), all models on the same patients:")
    st = subtype_wide(models); st.to_csv(f"{OUT}/step7_fusion_by_subtype.csv"); display(st)
    print("\nAUC per cohort:")
    sc = subtype_wide(models, "cohort"); sc.to_csv(f"{OUT}/step7_fusion_by_cohort.csv"); display(sc)

In [ ]:
# ==== PUSH the small result files to the vaib branch (results/<PERSON>/...) ====
if PUSH_RESULTS and not _tok:
    print("PUSH_RESULTS needs the GITHUB_TOKEN secret; download results/ from the Output tab instead")
elif PUSH_RESULTS:
    git("config", "user.name", f"Person {PERSON} (Kaggle)")
    git("config", "user.email", "kaggle@users.noreply.github.com")
    for attempt in range(3):
        git("fetch", "--quiet", "--depth", "1", "origin", f"+refs/heads/{GIT_TAG}:refs/remotes/origin/{GIT_TAG}")
        git("checkout", "--quiet", "-f", "-B", GIT_TAG, f"origin/{GIT_TAG}")
        shutil.copytree(f"/kaggle/working/results/{PERSON}", f"results/{PERSON}", dirs_exist_ok=True)
        git("add", f"results/{PERSON}")
        print(git("commit", "-m", f"results: {PERSON} {NOTEBOOK_NAME} ({GIT_TAG})"))
        try:
            git("push", "--quiet", "origin", GIT_TAG); print("pushed to", GIT_TAG); break
        except RuntimeError as e:
            print(f"push attempt {attempt + 1} failed:", e)
    else:
        print("NOT pushed: download results/ from the Output tab")